# tunekit — QLoRA fine-tuning in Colab

Fine-tunes an open LLM in 4-bit (QLoRA) on Colab's free T4, then evaluates, chats with, and exports it.

**Before you start: Runtime → Change runtime type → T4 GPU.** Without a GPU, 4-bit training cannot run
(bitsandbytes is CUDA-only) and the notebook will tell you so in step 0.

In [ ]:
#@title Step 0 — check the GPU and install tunekit
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun"
# Colab preinstalls torchao 0.10, which current peft rejects outright (ImportError at
# adapter-load time). tunekit does not use torchao, so removing it is the clean fix.
!pip uninstall -q -y torchao 2>/dev/null
!pip install -q "tunekit[quant]"
# 'bitsandbytes (QLoRA): available' below is the precondition for 4-bit training.
!tunekit info

## 1. Data

tunekit reads the first row and picks the converter, so these all work as-is:

| format | shape |
|---|---|
| messages | `{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}` |
| alpaca | `{"instruction": "...", "input": "...", "output": "..."}` |
| sharegpt | `{"conversations": [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]}` |
| prompt/completion | `{"prompt": "...", "completion": "..."}` |
| text | `{"text": "..."}` |

The cell below uses a small bundled example. Set `UPLOAD = True` to pick your own file instead, or
set `DATA` to a Hugging Face dataset id (e.g. `"yahma/alpaca-cleaned"`).

In [ ]:
UPLOAD = False   #@param {type:"boolean"}

if UPLOAD:
    from google.colab import files
    DATA = next(iter(files.upload()))          # blocks until you choose a file
else:
    !curl -sL -o pirate.jsonl https://raw.githubusercontent.com/Dolmaa24/tunekit/main/examples/pirate.jsonl
    DATA = "pirate.jsonl"
print("using:", DATA)

In [ ]:
#@title Step 2 — inspect the data before spending GPU time
# Shows the detected format, the exact text the model will train on, and token-length percentiles.
!tunekit validate --model Qwen/Qwen2.5-1.5B-Instruct --data {DATA}

## 3. Train in 4-bit (QLoRA)

`model.load_in_4bit=true` loads the base model as 4-bit NF4 and trains LoRA adapters on top. On a free T4:

| model | mode | VRAM |
|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | QLoRA 4-bit | ~3 GB |
| `Qwen/Qwen2.5-7B-Instruct` | QLoRA 4-bit | ~12 GB |
| `meta-llama/Llama-3.2-3B-Instruct` | QLoRA 4-bit | ~6 GB (needs a HF token + license acceptance) |

`tunekit info --model` reports the weight size against this GPU before anything downloads.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   #@param {type:"string"}

!tunekit info --model {MODEL}

!tunekit train --model {MODEL} --data {DATA} --output outputs/qlora \
    --set model.load_in_4bit=true \
    --set train.epochs=2 --set train.batch_size=4 --set train.grad_accum=4 \
    --set train.optimizer=paged_adamw_8bit --set data.max_length=1024

## 4. Did it help?

`tunekit eval` scores the held-out split with the tuned *and* the base model, and prints sample
generations side by side. Lower loss on the assistant turns plus better-looking samples is the signal.

In [ ]:
# --load-in-4bit reloads the base quantised, the way it was trained.
!tunekit eval outputs/qlora --samples 3 --max-new-tokens 80 --load-in-4bit

## 5. Chat with it from Python

In [ ]:
from tunekit.inference import load_for_inference, stream_reply

# load_in_4bit matches how it was trained; without it the base loads in bf16,
# which for a 7B adapter would not fit a T4.
model, tok = load_for_inference("outputs/qlora", load_in_4bit=True)

def ask(q, **kw):
    print("".join(stream_reply(model, tok, [{"role": "user", "content": q}], **kw)))

ask("What is the capital of France?", max_new_tokens=120)

## 6. Export (optional)

Merge the adapter into the base model, then convert to GGUF for Ollama / LM Studio / llama.cpp.

Note we install `tunekit[gguf]` rather than llama.cpp's own `requirements-convert_hf_to_gguf.txt`:
those pins would downgrade transformers and torch and break the environment above.

In [ ]:
!tunekit merge outputs/qlora outputs/merged --dtype float16
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q "tunekit[gguf]"
!tunekit export outputs/merged --quant q8_0 --llama-cpp llama.cpp --ollama
!ls -lh outputs/merged/*.gguf outputs/merged/Modelfile

In [ ]:
#@title Download the adapter (small — a few MB, unlike the merged model)
!zip -qr adapter.zip outputs/qlora -x "outputs/qlora/checkpoint-*"
from google.colab import files
files.download("adapter.zip")